# 97장 · 대화 턴의 현재 출처와 이전 기록

현재 화면의 답변과 출처는 새 질문에 맞게 바뀌지만 지난 턴의 근거는 기록에 남습니다. 두 저장 위치를 따로 갱신합니다.

장비 A 질문 다음에 교체 일정을 묻는 대화를 처리합니다. 새 답변의 출처는 비우고 직전 답변의 RULE-A 인용은 대화 이력에 보존합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 새 질문을 받으면 현재 카드만 비우기

96장의 카드를 첫 답의 기록으로 남긴 뒤, 규정에 없는 일정을 묻는 두 번째 요청을 받습니다. 지난 답과 현재 표시할 카드의 소유권을 구분합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter96.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter97-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter96.json"
sample_file = here / "chapter97-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['cards'])

교재 예시 chapter97-start.json
[{'id': 'RULE-A', 'answer': '점검 전 전원 표시등을 확인합니다.', 'citation': 'RULE-A | 가상 장비실 규정 A | p.1 | 2.1절 | 개정일 2026-11-01 | rules-v1'}]


두 번째 줄은 읽은 파일의 `cards` 값입니다.

## 질문 전환 시 현재 출처 교체

기능 A에서 받은 첫 답은 answered와 RULE-A였습니다. 기능 B의 일정 질문 Q03은 no_evidence와 빈 출처를 돌려줍니다. 화면 상태를 새 결과로 통째로 바꾸면 앞선 원문 링크가 새 질문의 근거로 보이지 않습니다. 아래 예시는 로컬 기록의 결과를 다시 사용합니다.

**현재 화면과 대화 이력**　왼쪽 화면에서 새 질문을 처리하는 동안 오른쪽 기록에는 이전 턴의 출처가 남습니다. 현재 화면을 지우는 코드가 대화 이력까지 바꾸는지 확인하세요.

### 셀 01 · 현재 턴의 출처 교체

화면에서 현재 답만 덮어쓰고 지난 답을 이력에 넣습니다.


In [1]:
first = {'status': 'answered',
 'answer': '점검 전 전원 표시등을 확인합니다.',
 'sources': [{'id': 'RULE-A',
              'text': '점검 전 전원 표시등을 확인합니다.',
              'source': '가상 장비실 규정 A',
              'page': 1,
              'section': '2.1',
              'revised_on': '2026-11-01',
              'version': 'rules-v1'}],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
second = {'status': 'no_evidence',
 'answer': '주어진 문서에서 확인할 수 없습니다. 담당자에게 확인하세요.',
 'sources': [],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
history = [first]
current = second
print(current["status"], current["sources"])
print(history[0]["sources"][0]["id"])

no_evidence []
RULE-A


현재 출처는 비고 이력에는 RULE-A가 그대로 있습니다. 새 질문의 화면에서는 출처가 사라지고, 앞선 질문의 이력에는 근거가 남습니다.

마지막의 RULE-A는 앞선 대화에 보관한 인용입니다. 현재 질문의 no_evidence 옆에 표시할 출처는 빈 목록입니다. 한 변수를 현재 패널과 과거 기록에 함께 쓰면 이전 출처를 잃거나 잘못된 근거를 노출하게 됩니다.


#### 지금 보이는 출처와 지난 대화 기록

<mark>현재 출처를 비워도 이미 답한 턴의 원문 링크는 남겨야 합니다.</mark> A를 물은 뒤 근거 없는 질문, B 질문을 차례로 보내세요. 이력에서는 A의 출처를 다시 열 수 있고 현재 카드에서는 B의 출처만 보여야 합니다.

선수 개념 · 35장 채팅 세션과 그래프의 대화 기록

## 이력 보존과 다음 답변 갱신

인계 카드　Q01 answered·RULE-A → Q03 no_evidence·인용 없음 → Q04 answered·RULE-B. 이 순서에는 근거 있는 답에서 없는 답으로 가는 경우와 다시 근거가 생기는 경우가 모두 있습니다. 팀 주제에서는 연속 질문을 보낸 다음 현재 화면과 접힌 과거 이력의 자료가 각각 어느 질문에 속하는지 표시하세요.


### 팀 참고 예시 · 이력과 현재 화면 분리


In [2]:
third = {'status': 'answered',
 'answer': '점검 전 고정 장치를 확인합니다.',
 'sources': [{'id': 'RULE-B',
              'text': '점검 전 고정 장치를 확인합니다.',
              'source': '가상 장비실 규정 B',
              'page': 2,
              'section': '2.2',
              'revised_on': '2026-11-01',
              'version': 'rules-v1'}],
 'mode': 'local_rule_fixture',
 'model_calls': 0}
updated_history = [*history, current]
current_reply = third
print(updated_history[-1]["status"], current_reply["sources"][0]["id"])

no_evidence RULE-B


no_evidence와 RULE-B가 나옵니다. 근거 없는 이전 답도 지우지 않아야 어떤 질문에서 문제가 생겼는지 다시 볼 수 있습니다.


#### 세 번째 질문에서 RULE-B는 어느 칸에 들어갈까?

```mermaid
flowchart TD
nsecond["Q03 현재"]
  nhistory["이력에 보관"]
  nthird["Q04 현재"]
  nsource["현재 출처"]
  nsecond --> nhistory
  nhistory --> nthird
  nthird --> nsource
```

Q01과 Q03을 지난 다음, 지금 표시할 답과 보관할 기록을 구분합니다. Q03의 빈 출처도 그 턴의 응답에 속합니다.

## 세 턴을 넘긴 뒤 과거 카드의 출처가 그대로인지 검사하기

사용자가 Q01 다음에 Q03을 묻고 마지막에 장비 B를 확인합니다. 화면에는 가장 최근 턴만 띄우고 지난 응답은 그때 반환된 내용으로 남겨야 합니다.

### 셀 04 · 과거 응답과 현재 응답을 별도 사본으로 보관하기

first, second, third에는 각각 A의 근거, 빈 근거, B의 근거가 있습니다. 한 화면의 선택 표시를 고쳐도 저장된 이전 카드가 바뀌지 않는지 확인하세요.

In [3]:
from copy import deepcopy

turns = [deepcopy(item) for item in (first, second, third)]
active_card = deepcopy(turns[-1])
active_card["sources"][0]["selected_on_screen"] = True
print([[s["id"] for s in row["sources"]] for row in turns])
print("저장된 카드의 선택 표시", "selected_on_screen" in turns[-1]["sources"][0])

[['RULE-A'], [], ['RULE-B']]
저장된 카드의 선택 표시 False


가운데 턴의 목록은 비고 마지막에만 RULE-B가 남습니다. 선택 표시는 화면 사본에만 추가해 저장된 카드에서는 False입니다. 바깥 사전만 복사하면 안쪽 sources 목록을 공유할 수 있습니다.

### 팀 상태에서 분리할 값 찾기

팀 화면에 접힌 상태, 펼친 절 또는 임시 입력이 있다면 한 항목을 바꾼 뒤 저장된 응답을 다시 읽으세요. 언제든 복구해야 하는 업무 기록과 창을 닫으면 버려도 되는 화면 설정의 예를 하나씩 적습니다.

<mark>과거 근거는 질문 시점의 기록이고 현재 화면의 선택은 바뀔 수 있습니다.</mark> Q03의 현재 출처를 이전 턴의 RULE-A로 메우면 답변 유보의 의미가 사라집니다.

98장에서는 입출력 기록을 유지한 채 검색을 준비하는 단계의 장비명 표기만 바꿉니다.

## 팀 사례에 적용하기 · 새 질문을 받으면 현재 카드만 비우기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
timeline = [{"question": "장비 A의 점검 전 확인 사항은?", "cards": list(incoming["cards"])},
            {"question": "장비 A의 교체 일정은?", "cards": []}]
new = {"quality_reference": incoming["quality_reference"], "turns": timeline, "prepared": incoming["prepared"]}
print([len(turn["cards"]) for turn in timeline])

[1, 0]


두 번째 답의 cards는 비어 있고 첫 번째 질문의 카드 수는 그대로입니다. 팀에서 세 번째 질문을 추가할 때에도 현재 선택과 이전 응답을 따로 보관할 수 있습니다.

In [ ]:
output_file = project_root / "team-project/final/chapter97.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter97.json


98장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

현재 화면은 마지막 요청 한 건의 답·출처로 바꾸고, 과거 기록은 별도로 보관하세요. 98장에서 질문 표기를 바꿔 실험할 때도 이전 응답의 출처가 새 응답에 섞이지 않는지 봅니다.
